# Treasury Carry Planner Validation

This notebook validates the new carry-planning view for the treasury account monitor.

It checks:
- account position table generation
- ZF futures-option carry table
- sorting by `dte direction strike delta price gamma`
- monthly target-return sizing

In [1]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import pandas as pd

try:
    display
except NameError:
    def display(value):
        print(value)

from target_treasury_account_monitor.option_chain_view import snapshot_to_monitor_frame
from target_treasury_account_monitor.carry_view import (
    account_positions_frame,
    capital_base_value,
    zf_option_carry_frame,
)

print("repo_root", repo_root)


repo_root e:\策略\IB-API


## 1. Offline ZF Option Carry Table

This uses synthetic rows shaped like `positions_to_frame()` output. No IB connection is needed.

In [ ]:
sample_frame = pd.DataFrame([
    {
        "symbol": "ZF",
        "secType": "FOP",
        "expiry": "20260709",
        "right": "P",
        "strike": 106.75,
        "delta": -0.25,
        "price": 0.131,
        "gamma": 0.010,
        "position": -2,
        "multiplier": 1000,
        "localSymbol": "ZF P 106.75",
        "iv": 0.11,
        "theta": 0.004,
        "marketValue": -262,
        "unrealizedPnL": 20,
    },
    {
        "symbol": "ZF",
        "secType": "FOP",
        "expiry": "20260709",
        "right": "C",
        "strike": 108.00,
        "delta": 0.20,
        "price": 0.100,
        "gamma": 0.020,
        "position": 0,
        "multiplier": 1000,
        "localSymbol": "ZF C 108",
        "iv": 0.10,
        "theta": 0.003,
        "marketValue": 0,
        "unrealizedPnL": 0,
    },
    {
        "symbol": "ZN",
        "secType": "FOP",
        "expiry": "20260709",
        "right": "P",
        "strike": 109.00,
        "delta": -0.30,
        "price": 0.200,
        "gamma": 0.030,
        "position": -1,
        "multiplier": 1000,
        "localSymbol": "ZN should be filtered",
    },
])

carry = zf_option_carry_frame(
    sample_frame,
    target_return=0.10,
    capital_base=100_000,
)
carry[[
    "dte",
    "direction",
    "strike",
    "delta",
    "price",
    "gamma",
    "signedDelta",
    "premiumPerContract",
    "targetMonthlyPremium",
    "contractsForTarget",
    "deltaAtTarget",
    "localSymbol",
]]

## 2. Assertions

Expected first row follows the user-facing sort example: `13 put 106.75 0.25 0.131 ...`.

In [ ]:
assert len(carry) == 2
first = carry.iloc[0]
assert first["direction"] == "put"
assert abs(float(first["strike"]) - 106.75) < 1e-9
assert abs(float(first["delta"]) - 0.25) < 1e-9
assert abs(float(first["signedDelta"]) + 0.25) < 1e-9
assert abs(float(first["premiumPerContract"]) - 131.0) < 1e-9
assert abs(float(first["targetMonthlyPremium"]) - 10_000.0) < 1e-9
assert abs(float(first["contractsForTarget"]) - (10_000 / 131.0)) < 1e-9
print("offline carry planner assertions passed")

## 3. Capital Base Resolution

Validate Net liquidation / Excess liquidity / Available funds / Custom base selection.

In [ ]:
summary = pd.DataFrame([
    {"tag": "NetLiquidation", "value": 100_000},
    {"tag": "ExcessLiquidity", "value": 80_000},
    {"tag": "AvailableFunds", "value": 60_000},
])

assert capital_base_value(summary, "Net liquidation", 0) == 100_000
assert capital_base_value(summary, "Excess liquidity", 0) == 80_000
assert capital_base_value(summary, "Available funds", 0) == 60_000
assert capital_base_value(summary, "Custom", 123_456) == 123_456
print("capital base assertions passed")

## 4. Account Position Sign Handling

Validate that a legitimate short quantity of `-1` is preserved instead of being treated as a missing IB sentinel.


In [ ]:
class FakeContract:
    conId = 1
    symbol = "ZF"
    localSymbol = "OZFN6 P1067"
    secType = "FOP"
    lastTradeDateOrContractMonth = "20260626"
    right = "P"
    strike = 106.75

class FakePosition:
    account = "U_TEST"
    contract = FakeContract()
    position = -1
    avgCost = 100

account_test = account_positions_frame([FakePosition()], {})
assert float(account_test.iloc[0]["position"]) == -1.0
print("account position sign assertion passed")


## 5. Optional Live IB Snapshot

Set `RUN_LIVE_IB_TEST = True` after IB Gateway/TWS is running. This pulls the real account positions and renders the same carry table used by the Streamlit tab.

In [2]:
RUN_LIVE_IB_TEST = True

if RUN_LIVE_IB_TEST:
    from target_treasury_account_monitor.config import (
        DEFAULT_CLIENT_ID,
        DEFAULT_HOST,
        DEFAULT_PORT,
        MARKET_DATA_TYPES,
        MonitorSettings,
    )
    from target_treasury_account_monitor.ib_client import (
        account_summary_frame,
        fetch_target_positions,
        get_future_reference,
        portfolio_items_by_key,
        refresh_account_portfolio,
        subscribe_quotes_for_positions,
    )
    from target_treasury_account_monitor.frames import positions_to_frame
    from target_treasury_account_monitor.carry_view import account_positions_frame
    from ib_async import IB, util

    ACCOUNT = "U16251798"  # Fill this with your account id, e.g. U1234567.
    assert ACCOUNT, "Fill ACCOUNT before running live test."

    settings = MonitorSettings(
        host=DEFAULT_HOST,
        port=DEFAULT_PORT,
        client_id=DEFAULT_CLIENT_ID + 80,
        account=ACCOUNT,
        market_data_type=MARKET_DATA_TYPES["Delayed"],
        quote_wait_seconds=6,
        refresh_seconds=5,
        auto_refresh=False,
        auto_reconnect=False,
        reconnect_backoff_seconds=10,
        wechat_webhook_url="",
        wechat_push_enabled=False,
        wechat_min_interval_seconds=300,
    )

    util.startLoop()
    ib = IB()
    ib.connect(settings.host, settings.port, clientId=settings.client_id, readonly=True, timeout=10)
    try:
        positions, all_positions = fetch_target_positions(ib, settings.account)
        future_ref = get_future_reference(ib, positions, settings, root="ZF")
        tickers = subscribe_quotes_for_positions(ib, positions, settings)
        refresh_account_portfolio(ib, settings.account)
        portfolio_map = portfolio_items_by_key(ib, settings.account)
        position_frame = positions_to_frame(
            positions,
            tickers,
            portfolio_map,
            reference_price=float(future_ref.get("price", float("nan"))),
        )
        summary_live = account_summary_frame(ib, settings.account)
        account_frame = account_positions_frame(all_positions, portfolio_map)
        live_carry = zf_option_carry_frame(
            position_frame,
            target_return=0.10,
            capital_base=capital_base_value(summary_live, "Net liquidation", 0),
        )
    finally:
        ib.disconnect()

    display(account_frame)
    display(live_carry)
else:
    print("live IB test skipped")

Error 200, reqId 3: No security definition has been found for the request, contract: Future(symbol='ZF', lastTradeDateOrContractMonth='202607', exchange='CBOT', currency='USD')
Unknown contract: Future(symbol='ZF', lastTradeDateOrContractMonth='202607', exchange='CBOT', currency='USD')


,account,symbol,localSymbol,secType,expiry,direction,strike,position,avgCost,marketPrice,marketValue,unrealizedPnL,realizedPnL,conId
0,U16251798,MCL,MCOQ6 C7900,FOP,20260716,C,79.00,1.0,253.020000,0.570000,57.00,-196.02,0.0,800566747
1,U16251798,ZF,VF5M6 P1062,FOP,20260629,P,106.25,-2.0,29.530000,0.000885,-1.77,57.29,0.0,892359583
2,U16251798,ZF,VF5M6 P1065,FOP,20260629,P,106.50,-1.0,45.160000,0.001877,-1.88,43.28,0.0,892359507
3,U16251798,ZF,VF5M6 P1067,FOP,20260629,P,106.75,-2.0,33.435000,0.003202,-6.40,60.47,0.0,892359430
4,U16251798,ZF,VF5M6 P1072,FOP,20260629,P,107.25,-1.0,76.410000,0.048838,-48.84,27.57,0.0,892359415
5,U16251798,ZF,VF5M6 C1075,FOP,20260629,C,107.50,-4.0,21.720000,0.005914,-23.65,63.23,0.0,892359550
6,U16251798,ZF,GF5M6 P1065,FOP,20260630,P,106.50,-1.0,52.970000,0.002884,-2.88,50.09,0.0,892750952
7,U16251798,ZF,GF5M6 P1070,FOP,20260630,P,107.00,-3.0,68.593333,0.020416,-61.25,144.53,0.0,892750751
8,U16251798,ZF,GF5M6 C1072,FOP,20260630,C,107.25,-2.0,13.910000,0.095969,-191.94,-164.12,0.0,892750794
9,U16251798,ZF,GF5M6 C1075,FOP,20260630,C,107.50,-3.0,26.925000,0.023438,-70.31,10.46,0.0,892750818


,dte,direction,strike,delta,price,gamma,signedDelta,position,premiumPerContract,currentCarryPremium,...,expiry,iv,theta,greekReady,tradeCandidateReady,priceSource,greekSource,missingData,marketValue,unrealizedPnL
0,0.0,put,106.25,NaN,0.000885,NaN,NaN,-2.0,0.88500,1.77000,...,20260629,NaN,NaN,False,False,portfolio,,greeks unavailable,-1.77,57.29
1,0.0,put,106.50,NaN,0.001877,NaN,NaN,-1.0,1.87705,1.87705,...,20260629,NaN,NaN,False,False,portfolio,,greeks unavailable,-1.88,43.28
2,0.0,put,106.75,NaN,0.003202,NaN,NaN,-2.0,3.20190,6.40380,...,20260629,NaN,NaN,False,False,portfolio,,greeks unavailable,-6.40,60.47
3,0.0,put,107.25,0.499702,0.048838,2.713406,-0.499702,-1.0,48.83760,48.83760,...,20260629,0.035131,-0.058947,True,True,portfolio,modelGreeks,,-48.84,27.57
4,0.0,call,107.50,0.078184,0.005914,0.829856,0.078184,-4.0,5.91375,23.65500,...,20260629,0.041577,-0.006251,True,True,portfolio,modelGreeks,,-23.65,63.23
5,1.0,put,106.50,NaN,0.002884,NaN,NaN,-1.0,2.88350,2.88350,...,20260630,NaN,NaN,False,False,portfolio,,greeks unavailable,-2.88,50.09
6,1.0,put,107.00,0.155681,0.020416,0.967050,-0.155681,-3.0,20.41555,61.24665,...,20260630,0.035010,-0.017442,True,True,portfolio,modelGreeks,,-61.25,144.53
7,1.0,call,107.25,0.500385,0.095969,1.657367,0.500385,-2.0,95.96885,191.93770,...,20260630,0.033632,-0.038798,True,True,portfolio,modelGreeks,,-191.94,-164.12
8,1.0,call,107.50,0.158081,0.023438,0.971612,0.158081,-3.0,23.43750,70.31250,...,20260630,0.034929,-0.017787,True,True,portfolio,modelGreeks,,-70.31,10.46
9,1.0,call,107.75,0.041763,0.006011,0.309240,0.041763,-2.0,6.01125,12.02250,...,20260630,0.042685,-0.004836,True,True,portfolio,modelGreeks,,-12.02,15.80


In [ ]:
if "live_carry" in globals():
    # live_carry.to_excel("live_carry.xlsx", index=False)
    live_carry.to_csv("live_carry.csv", index=False)
    print("wrote live_carry.xlsx")
else:
    print("live_carry export skipped")


PermissionError: [Errno 13] Permission denied: 'live_carry.xlsx'

## 5. Offline Option Chain Candidate Conversion

This validates the option-chain snapshot normalization path. It does not require IB.


In [ ]:
chain_snapshot = pd.DataFrame([
    {
        "snapshotTimeUtc": "2026-06-26T00:00:00Z",
        "conId": 101,
        "symbol": "ZF",
        "localSymbol": "OZFN6 P1067",
        "expiration": "20260709",
        "strike": 106.75,
        "right": "P",
        "bid": 0.125,
        "ask": 0.137,
        "last": None,
        "markPrice": None,
        "close": None,
        "openInterest": 100,
        "iv": 0.12,
        "delta": -0.25,
        "gamma": 0.01,
        "theta": -0.004,
        "modelGreeks_delta": -0.25,
    },
    {
        "snapshotTimeUtc": "2026-06-26T00:00:00Z",
        "conId": 102,
        "symbol": "ZF",
        "localSymbol": "OZFN6 C1070",
        "expiration": "20260709",
        "strike": 107.00,
        "right": "C",
        "bid": 0.090,
        "ask": 0.110,
        "last": None,
        "markPrice": None,
        "close": None,
        "openInterest": 80,
        "iv": 0.11,
        "delta": 0.31,
        "gamma": 0.012,
        "theta": -0.005,
        "modelGreeks_delta": 0.31,
    },
])

chain_monitor_frame = snapshot_to_monitor_frame(chain_snapshot, root="ZF")
chain_carry = zf_option_carry_frame(
    chain_monitor_frame,
    target_return=0.10,
    capital_base=100_000,
)

assert len(chain_carry) == 2
assert chain_carry.loc[0, "direction"] == "put"
assert abs(chain_carry.loc[0, "price"] - 0.131) < 1e-9
assert chain_carry["contractsForTarget"].notna().all()
print("offline option-chain candidate assertions passed")
chain_carry


## 6. Optional Live ZF Option Chain Probe

Set `RUN_LIVE_CHAIN_TEST = True` only when IB Gateway/TWS is connected. Use `FULL_CHAIN_SNAPSHOT = True` when you really want to subscribe the entire qualified chain; it can be slow and may hit IB pacing or market-data limits.


In [7]:
RUN_LIVE_CHAIN_TEST = True
FULL_CHAIN_SNAPSHOT = True

HOST = "127.0.0.1"
PORT = 4001
CLIENT_ID = 861
MARKET_DATA_TYPE = 3
MIN_MONTH = "202609"
MANUAL_FUTURE_MONTHS = "202609,202612"

if RUN_LIVE_CHAIN_TEST:
    import importlib
    from ib_async import IB, util
    from ib_async.ib import StartupFetch
    import target_treasury_account_monitor.option_chain_view as option_chain_view

    importlib.reload(option_chain_view)
    fetch_zf_option_chain_snapshot = option_chain_view.fetch_zf_option_chain_snapshot

    util.startLoop()
    ib = IB()
    ib.connect(HOST, PORT, clientId=CLIENT_ID, readonly=True, timeout=10, fetchFields=StartupFetch(0))
    try:
        result = fetch_zf_option_chain_snapshot(
            ib,
            root="ZF",
            market_data_type=MARKET_DATA_TYPE,
            future_months=MANUAL_FUTURE_MONTHS,
            min_month=MIN_MONTH,
            max_future_months=2,
            full_chain_snapshot=FULL_CHAIN_SNAPSHOT,
            dte0_width=2.0,
            non_dte0_width=5.0,
            batch_size=150,
            wait_max_seconds=12.0,
            wait_stable_seconds=2.0,
        )
        print("months", result["months"])
        print("universe source", result["universe_source"])
        print("cache", result["cache_path"])
        print("universe contracts", result["contract_count"])
        print("snapshot contracts", result["snapshot_count"])
        display(result["future_prices"])
        display(result["selected_metadata"].head(20))
        display(result["monitor_frame"].head(20))
        display(zf_option_carry_frame(result["monitor_frame"], target_return=0.10, capital_base=100_000).head(50))
    finally:
        ib.disconnect()
else:
    print("live ZF option-chain test skipped")



market data batch 1: 150 contracts
sent 150/150 market data requests in 4.3s
ready quote=150/150, greeks=150/150, oi=150/150, volume=0/150, elapsed=4.4s
finished 150/568

market data batch 2: 150 contracts
sent 150/150 market data requests in 4.5s
ready quote=150/150, greeks=150/150, oi=150/150, volume=0/150, elapsed=5.9s
finished 300/568

market data batch 3: 150 contracts
sent 150/150 market data requests in 4.5s
ready quote=150/150, greeks=150/150, oi=150/150, volume=0/150, elapsed=6.1s
finished 450/568

market data batch 4: 118 contracts
sent 118/118 market data requests in 3.6s
ready quote=118/118, greeks=118/118, oi=118/118, volume=0/118, elapsed=6.7s
finished 568/568
months ['202609', '202612']
universe source cache_partial_fallback
cache ZF_FOP_Universe_202606_202609_202612.csv
universe contracts 568
snapshot contracts 568


,root,month,exchange,conId,localSymbol,price,priceSource,marketPrice,mid,last,close,bid,ask
0,ZF,202609,CBOT,842590380,ZFU6,107.265625,marketPrice,107.265625,107.261719,107.265625,107.31250,107.257812,107.265625
1,ZF,202612,CBOT,869928209,ZFZ6,107.238281,marketPrice,107.238281,107.238281,NaN,107.28125,107.218750,107.257812


,underlyingConId,underlyingLocalSymbol,underlyingMonth,chainExchange,chainTradingClass,chainMultiplier,conId,secType,symbol,localSymbol,tradingClass,lastTradeDateOrContractMonth,strike,right,multiplier,exchange,currency
0,842590380,ZFU6,202609,CBOT,OZF,1000,884727137,FOP,ZF,OZFQ6 C0990,OZF,20260724,99.00,C,1000,CBOT,USD
1,842590380,ZFU6,202609,CBOT,OZF,1000,884600757,FOP,ZF,OZFQ6 C0992,OZF,20260724,99.25,C,1000,CBOT,USD
2,842590380,ZFU6,202609,CBOT,OZF,1000,884600762,FOP,ZF,OZFQ6 C0995,OZF,20260724,99.50,C,1000,CBOT,USD
3,842590380,ZFU6,202609,CBOT,OZF,1000,882371990,FOP,ZF,OZFQ6 C0997,OZF,20260724,99.75,C,1000,CBOT,USD
4,842590380,ZFU6,202609,CBOT,OZF,1000,869663191,FOP,ZF,OZFQ6 C1000,OZF,20260724,100.00,C,1000,CBOT,USD
5,842590380,ZFU6,202609,CBOT,OZF,1000,869663112,FOP,ZF,OZFQ6 C1002,OZF,20260724,100.25,C,1000,CBOT,USD
6,842590380,ZFU6,202609,CBOT,OZF,1000,869663066,FOP,ZF,OZFQ6 C1005,OZF,20260724,100.50,C,1000,CBOT,USD
7,842590380,ZFU6,202609,CBOT,OZF,1000,869662835,FOP,ZF,OZFQ6 C1007,OZF,20260724,100.75,C,1000,CBOT,USD
8,842590380,ZFU6,202609,CBOT,OZF,1000,869662911,FOP,ZF,OZFQ6 C1010,OZF,20260724,101.00,C,1000,CBOT,USD
9,842590380,ZFU6,202609,CBOT,OZF,1000,869663111,FOP,ZF,OZFQ6 C1012,OZF,20260724,101.25,C,1000,CBOT,USD


,snapshotTimeUtc,marketDataType,conId,symbol,localSymbol,tradingClass,expiration,strike,right,exchange,...,modelGreeks_undPrice,modelGreeks_pvDividend,price,priceSource,expiry,secType,position,multiplier,greekSource,missingData
0,2026-06-29T06:05:11.344687+00:00,1,884727137,ZF,OZFQ6 C0990,OZF,20260724,99.00,C,CBOT,...,107.265625,0.0,8.265625,mid,20260724,FOP,0.0,1000.0,modelGreeks,
1,2026-06-29T06:05:11.344687+00:00,1,884600757,ZF,OZFQ6 C0992,OZF,20260724,99.25,C,CBOT,...,107.265625,0.0,8.015625,mid,20260724,FOP,0.0,1000.0,modelGreeks,
2,2026-06-29T06:05:11.344687+00:00,1,884600762,ZF,OZFQ6 C0995,OZF,20260724,99.50,C,CBOT,...,107.265625,0.0,7.765625,mid,20260724,FOP,0.0,1000.0,modelGreeks,
3,2026-06-29T06:05:11.344687+00:00,1,882371990,ZF,OZFQ6 C0997,OZF,20260724,99.75,C,CBOT,...,107.265625,0.0,7.515625,mid,20260724,FOP,0.0,1000.0,modelGreeks,
4,2026-06-29T06:05:11.344687+00:00,1,869663191,ZF,OZFQ6 C1000,OZF,20260724,100.00,C,CBOT,...,107.265625,0.0,7.265625,mid,20260724,FOP,0.0,1000.0,modelGreeks,
5,2026-06-29T06:05:11.344687+00:00,1,869663112,ZF,OZFQ6 C1002,OZF,20260724,100.25,C,CBOT,...,107.265625,0.0,7.015625,mid,20260724,FOP,0.0,1000.0,modelGreeks,
6,2026-06-29T06:05:11.344687+00:00,1,869663066,ZF,OZFQ6 C1005,OZF,20260724,100.50,C,CBOT,...,107.265625,0.0,6.765625,mid,20260724,FOP,0.0,1000.0,modelGreeks,
7,2026-06-29T06:05:11.344687+00:00,1,869662835,ZF,OZFQ6 C1007,OZF,20260724,100.75,C,CBOT,...,107.265625,0.0,6.515625,mid,20260724,FOP,0.0,1000.0,modelGreeks,
8,2026-06-29T06:05:11.344687+00:00,1,869662911,ZF,OZFQ6 C1010,OZF,20260724,101.00,C,CBOT,...,107.265625,0.0,6.265625,mid,20260724,FOP,0.0,1000.0,modelGreeks,
9,2026-06-29T06:05:11.344687+00:00,1,869663111,ZF,OZFQ6 C1012,OZF,20260724,101.25,C,CBOT,...,107.265625,0.0,6.015625,mid,20260724,FOP,0.0,1000.0,modelGreeks,


,dte,direction,strike,delta,price,gamma,signedDelta,position,premiumPerContract,currentCarryPremium,...,deltaAtTarget,localSymbol,expiry,iv,theta,greekReady,tradeCandidateReady,priceSource,greekSource,missingData
0,25.0,put,99.00,0.000008,0.000000,0.000017,-0.000008,0.0,0.00000,0.0,...,inf,OZFQ6 P0990,20260724,0.071561,-0.000001,True,True,close,modelGreeks,
1,25.0,put,99.25,0.000018,0.000000,0.000038,-0.000018,0.0,0.00000,0.0,...,inf,OZFQ6 P0992,20260724,0.072183,-0.000003,True,True,close,modelGreeks,
2,25.0,put,99.50,0.000036,0.000000,0.000073,-0.000036,0.0,0.00000,0.0,...,inf,OZFQ6 P0995,20260724,0.072586,-0.000006,True,True,close,modelGreeks,
3,25.0,put,99.75,0.000065,0.000000,0.000126,-0.000065,0.0,0.00000,0.0,...,inf,OZFQ6 P0997,20260724,0.072780,-0.000010,True,True,close,modelGreeks,
4,25.0,put,100.00,0.000110,0.000000,0.000207,-0.000110,0.0,0.00000,0.0,...,inf,OZFQ6 P1000,20260724,0.072778,-0.000014,True,True,close,modelGreeks,
5,25.0,put,100.25,0.000184,0.000000,0.000337,-0.000184,0.0,0.00000,0.0,...,inf,OZFQ6 P1002,20260724,0.072594,-0.000030,True,True,close,modelGreeks,
6,25.0,put,100.50,0.000268,0.000000,0.000483,-0.000268,0.0,0.00000,0.0,...,inf,OZFQ6 P1005,20260724,0.072245,-0.000030,True,True,close,modelGreeks,
7,25.0,put,100.75,0.000426,0.000000,0.000746,-0.000426,0.0,0.00000,0.0,...,inf,OZFQ6 P1007,20260724,0.071744,-0.000060,True,True,close,modelGreeks,
8,25.0,put,101.00,0.000587,0.000000,0.001018,-0.000587,0.0,0.00000,0.0,...,inf,OZFQ6 P1010,20260724,0.071105,-0.000065,True,True,close,modelGreeks,
9,25.0,put,101.25,0.000874,0.000000,0.001479,-0.000874,0.0,0.00000,0.0,...,inf,OZFQ6 P1012,20260724,0.070338,-0.000115,True,True,close,modelGreeks,
